In [1]:
import os 
current_work_dir = os.getcwd()

In [19]:
import importlib
import sys

# Add the directory to sys.path if it's not already there
sys.path.append(current_work_dir)

# Import the module first
import tft_setup
import tft_interpret
import tft_interpret_updated

# Now reload it
importlib.reload(tft_setup)
importlib.reload(tft_interpret)

<module 'tft_interpret' from 'c:\\Users\\G0004878\\Desktop\\TFT_Data\\Daily_forecasting_model\\Iterations in September\\_Iteration_#2_calendar_attributes\\Interpretation\\tft_interpret.py'>

In [24]:
from tft_interpret_updated import extract_tft_interpretation,stratified_indices

In [21]:
import os 
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [23]:
from tft_setup import load_everything

In [6]:
ctx = load_everything(model_name="daily_tft_negbin_scooters_2026-09-15_20_02_52",work_dir=r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#2_calendar_attributes\Modelling",base_dir=r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in September\_Iteration_#2_calendar_attributes\Modelling",skip_dead_filter=True)

model folder contents: ['checkpoints', '_model.pth.tar']
model            : daily_tft_negbin_scooters_2026-09-15_20_02_52
ICL / OCL        : 365 / 98
series in manifest after filter : 31,931
series with val strip           : 31,931
history ends     : 2026-08-31  (forecast starts next day)
calendar covers  : 2023-04-01 -> 2026-12-07


In [14]:
ctx.val_statics[0]

,PARENT_DEALER_CODE,MODEL_FAMILY,MODEL_NAME,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME
0,11132,XTREME 125,XTREME 125,DRUM,SELF,CAST,GREY,TATHAWADE,URBAN,Zonal Office - West


In [12]:
import torch, pandas as pd

st = pd.concat(ctx.val_statics, ignore_index=True)
emb = [(n, m.num_embeddings) for n, m in ctx.model.model.named_modules()
       if isinstance(m, torch.nn.Embedding)]
print("static columns in data      :", st.shape[1])
print("embedding tables in model   :", len(emb))
print(emb)

k = max(st.shape[1], len(emb))
chk = pd.DataFrame({
    "column":         list(st.columns)         + [None] * (k - st.shape[1]),
    "categories":     list(st.nunique().values) + [None] * (k - st.shape[1]),
    "embedding_rows": [r for _, r in emb]      + [None] * (k - len(emb)),
})
chk["expected_rows"] = chk["categories"] + 1
chk["ok"] = chk["embedding_rows"] == chk["expected_rows"]
print(chk)

static columns in data      : 10
embedding tables in model   : 0
[]
               column  categories embedding_rows  expected_rows     ok
0  PARENT_DEALER_CODE        1098           None           1099  False
1        MODEL_FAMILY          13           None             14  False
2          MODEL_NAME          13           None             14  False
3          BRAKE_TYPE           2           None              3  False
4       IGNITION_TYPE           2           None              3  False
5          WHEEL_TYPE           4           None              5  False
6              COLOUR          27           None             28  False
7         DEALER_CITY         836           None            837  False
8     X_CITY_CATEGORY           3           None              4  False
9   ZONAL_OFFICE_NAME           5           None              6  False


In [15]:
tm = ctx.model.model
print("static vars     :", len(tm.static_variables), tm.static_variables)
print("categorical     :", getattr(tm, "categorical_static_variables", None))
print("numeric         :", getattr(tm, "numeric_static_variables", None))

static vars     : 10 ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5', 'static_covariate_6', 'static_covariate_7', 'static_covariate_8', 'static_covariate_9']
categorical     : []
numeric         : ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5', 'static_covariate_6', 'static_covariate_7', 'static_covariate_8', 'static_covariate_9']


In [25]:
labels = ctx.labels()
idx = stratified_indices(labels, n_per_group=100)
out = extract_tft_interpretation(
    ctx.model, ctx.val_seq, future_covariates=ctx.val_cov_seq,
    indices=idx, group_labels=labels, target_names=[ctx.target_col],
    future_names=ctx.future_covariates, static_names=ctx.static_covariates,
)
print(out["ALL"]["n_series"])

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[encoder] raw variable names (59): ['target_0', 'future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4'] ...
[decoder] raw variable names (58): ['future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4', 'future_covariate_5'] ...
[static] raw variable names (10): ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5'] ...


c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Generating TimeSeries:   0%|          | 0/1224 [00:00<?, ?it/s]

1224


In [28]:
import pandas as pd

sc = ctx.shared_cov
cov = pd.DataFrame(sc.values(), index=sc.time_index, columns=sc.components).loc["2026-09-01":"2026-12-07"]
assert len(cov) == 98, len(cov)          # must equal the 98 decoder steps

dt = out["ALL"]["decoder_time"]
for v in ["D-2", "N+3", "D", "IS_MONTH_START"]:
    a = (cov[v] != 0).to_numpy()
    print(f"{v:15s} active: {dt[v].to_numpy()[a].mean():.3f}   zero: {dt[v].to_numpy()[~a].mean():.3f}")

D-2             active: 0.176   zero: 0.104
N+3             active: 0.135   zero: 0.054
D               active: 0.204   zero: 0.047
IS_MONTH_START  active: 0.014   zero: 0.040


In [29]:
idx2 = stratified_indices(labels, n_per_group=100, seed=1)
out2 = extract_tft_interpretation(ctx.model, ctx.val_seq, future_covariates=ctx.val_cov_seq,
        indices=idx2, group_labels=labels, target_names=[ctx.target_col],
        future_names=ctx.future_covariates, static_names=ctx.static_covariates)
a = out["ALL"]["decoder_importance"]; b = out2["ALL"]["decoder_importance"].reindex(a.index)
print(pd.concat([a.rename("sample0"), b.rename("sample1")], axis=1).head(10))
print("rank correlation:", a.rank().corr(b.rank()))

[encoder] raw variable names (59): ['target_0', 'future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4'] ...
[decoder] raw variable names (58): ['future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4', 'future_covariate_5'] ...
[static] raw variable names (10): ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5'] ...


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Generating TimeSeries:   0%|          | 0/1224 [00:00<?, ?it/s]

                 sample0   sample1
D-2             0.104348  0.105666
DOW_SIN         0.066087  0.065192
N+3             0.054874  0.053779
D               0.048385  0.049587
IS_MONTH_START  0.036915  0.036397
N+9             0.034423  0.034147
D-3             0.033273  0.033018
D+3             0.032956  0.032917
IS_MONTH_END    0.028953  0.028223
AKSHYA_TRITIYA  0.027473  0.028119
rank correlation: 0.998400442954259


In [30]:
fam = "XOOM"
idx_f = [i for i in idx if labels[i] == fam]
solo = extract_tft_interpretation(ctx.model, ctx.val_seq, future_covariates=ctx.val_cov_seq,
        indices=idx_f, group_labels=labels, target_names=[ctx.target_col],
        future_names=ctx.future_covariates, static_names=ctx.static_covariates)
s = solo["ALL"]["decoder_importance"]
print("max abs difference:", (s - out[fam]["decoder_importance"].reindex(s.index)).abs().max())

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


[encoder] raw variable names (59): ['target_0', 'future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4'] ...
[decoder] raw variable names (58): ['future_covariate_0', 'future_covariate_1', 'future_covariate_2', 'future_covariate_3', 'future_covariate_4', 'future_covariate_5'] ...
[static] raw variable names (10): ['static_covariate_0', 'static_covariate_1', 'static_covariate_2', 'static_covariate_3', 'static_covariate_4', 'static_covariate_5'] ...


c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting: |          | 0/? [00:00<?, ?it/s]

Generating TimeSeries:   0%|          | 0/100 [00:00<?, ?it/s]

max abs difference: 2.5446013907082232e-09


In [31]:
import numpy as np
idx_s = stratified_indices(labels, n_per_group=20)
base  = [ctx.val_seq[i] for i in idx_s]
cov   = [ctx.val_cov_seq[i] for i in idx_s]

def totals(series_list, seed):
    p = ctx.model.predict(n=98, series=series_list, future_covariates=cov,
                          num_samples=100, random_state=seed, verbose=False)
    return np.array([x.all_values(copy=False).mean(axis=2).sum() for x in p])

def shuffled(col, seed=0):
    codes = np.array([s.static_covariates[col].iloc[0] for s in base])
    perm = np.random.default_rng(seed).permutation(codes)
    out = []
    for s, c in zip(base, perm):
        sc = s.static_covariates.copy(); sc[col] = c
        out.append(s.with_static_covariates(sc))
    return out

t0, t0b = totals(base, 0), totals(base, 1)
m = t0 > 1
print("noise floor (same input, different draw): %.1f%%" % (100*np.median(np.abs(t0b-t0)[m]/t0[m])))
for col in ["MODEL_FAMILY", "PARENT_DEALER_CODE", "COLOUR", "DEALER_CITY"]:
    t1 = totals(shuffled(col), 0)
    print(f"{col:20s} median change in 98-day total: {100*np.median(np.abs(t1-t0)[m]/t0[m]):.1f}%")

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


noise floor (same input, different draw): 3.3%


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


MODEL_FAMILY         median change in 98-day total: 4.3%


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


PARENT_DEALER_CODE   median change in 98-day total: 0.5%


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


COLOUR               median change in 98-day total: 13.0%
DEALER_CITY          median change in 98-day total: 0.4%
